# PARCIAL 1 MBA - Ejercicio 2
## Simulación de Tráfico en una Ciudad con Semáforos

**Objetivo:** Modelar el flujo de vehículos en una intersección de 4 calles.
Cada vehículo es un agente que se desplaza en su carril (Norte, Sur, Este u
Oeste). Dos semáforos coordinados controlan quién puede cruzar la
intersección: cuando el eje horizontal tiene verde, el vertical tiene rojo
y viceversa. Los carros respetan la fila: el de atrás no avanza si el de
adelante está detenido o demasiado cerca.

## Celda 1 - Instalación de dependencias
Instala la librería `mesa` (framework ABM) que no viene preinstalada en Google Colab.

In [ ]:
# Instalamos mesa (framework de modelado basado en agentes)
!pip install mesa --quiet

## Celda 2 - Importaciones y configuración de matplotlib para Colab

In [ ]:
# Configuramos matplotlib para renderizar animaciones en el notebook.
# embed_limit define el tamaño máximo (en MB) del HTML generado por to_jshtml().
# Con 120 frames la animación puede pesar ~30-40 MB, por lo que usamos 100 MB.
# Debe ejecutarse ANTES de importar pyplot.
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 100  # MB máximos para la animación HTML

# ── Librería Mesa: base para el modelado basado en agentes ──────────────────
from mesa import Agent, Model                    # Agent: agente base; Model: modelo base
from mesa.datacollection import DataCollector    # Recolector de estadísticas por paso

# ── Matplotlib: visualización y animación ───────────────────────────────────
import matplotlib.pyplot as plt                  # Construcción de la figura
import matplotlib.patches as mpatches            # Rectángulos y parches visuales
from matplotlib.animation import FuncAnimation   # Animación cuadro a cuadro

# ── Utilidades estándar ──────────────────────────────────────────────────────
import random                                    # Decisiones y posiciones aleatorias
import numpy as np                               # Arrays para set_offsets del scatter

# ── Necesario para mostrar la animación HTML dentro del notebook de Colab ───
from IPython.display import HTML

print("Dependencias importadas correctamente.")

## Celda 3 - Constantes globales

In [ ]:
# =============================================================================
# CONSTANTES GLOBALES — configuración central de la simulación
# =============================================================================

# ── Semáforos ────────────────────────────────────────────────────────────────

# Número de pasos que un semáforo permanece en verde antes de pasar a amarillo.
# Debe ser >= tiempo de cruce: la intersección mide 0.30 y la velocidad es 0.02,
# por lo que cruzar tarda 0.30 / 0.02 = 15 pasos. Con 40 pasos de verde hay
# margen amplio para que todos los carros en cola crucen cómodamente, y el
# semáforo se percibe visualmente como lento y realista.
CICLO_VERDE = 40

# Número de pasos en amarillo (advertencia: los carros frenan y se detienen).
# 5 pasos da tiempo visual suficiente para notar el cambio de color claramente.
CICLO_AMARILLO = 5

# Número de pasos en rojo (el eje contrario tiene verde durante este tiempo).
# Simétrico con CICLO_VERDE para que ambos ejes reciban el mismo tiempo de paso.
CICLO_ROJO = CICLO_VERDE  # Ambos ejes tienen el mismo tiempo de verde/rojo

# ── Carros ────────────────────────────────────────────────────────────────────

# Avance de cada carro por paso de simulación en coordenadas normalizadas [0, 1].
VELOCIDAD_CARRO = 0.02

# Distancia mínima que debe haber entre dos carros consecutivos del mismo carril.
# Si el carro de adelante está más cerca que esto, el de atrás se detiene.
DISTANCIA_MIN = 0.10

# Posición (en coordenadas [0, 1]) donde los carros frenan al encontrar semáforo
# en rojo o amarillo. Debe ser ESTRICTAMENTE MENOR que INTER_MIN (0.35).
STOP_LINE = 0.33

# Cada cuántos pasos de simulación aparece un nuevo carro en cada extremo.
SPAWN_INTERVALO = 20

# Máximo de carros por carril (por dirección).
MAX_CARROS_POR_CARRIL = 3

# ── Geometría de la intersección ─────────────────────────────────────────────

# Coordenadas del cuadrado central que representa la intersección.
INTER_MIN = 0.35   # Borde izquierdo/inferior de la intersección
INTER_MAX = 0.65   # Borde derecho/superior de la intersección

# ── Colores de los carros por dirección ─────────────────────────────────────
COLOR_CARRO = {
    "NORTE": "#E74C3C",   # Rojo: carros que vienen del Sur y van al Norte
    "SUR":   "#3498DB",   # Azul: carros que vienen del Norte y van al Sur
    "ESTE":  "#2ECC71",   # Verde: carros que vienen del Oeste y van al Este
    "OESTE": "#F39C12",   # Naranja: carros que vienen del Este y van al Oeste
}

# ── Estados posibles de un semáforo ─────────────────────────────────────────
VERDE    = "VERDE"
AMARILLO = "AMARILLO"
ROJO     = "ROJO"

print("Constantes configuradas correctamente.")
print(f"  Ciclo semáforo: {CICLO_VERDE}v / {CICLO_AMARILLO}a / {CICLO_ROJO}r pasos")
print(f"  Velocidad carro: {VELOCIDAD_CARRO} unidades/paso")
print(f"  Intersección: [{INTER_MIN}, {INTER_MAX}]")

## Celda 4 - Clase Semaforo

In [ ]:
# =============================================================================
# CLASE: Semaforo
# =============================================================================

class Semaforo:
    """
    Representa un semáforo que controla un eje de tráfico (horizontal o vertical).

    El ciclo de estados es:
        VERDE (CICLO_VERDE pasos)
          → AMARILLO (CICLO_AMARILLO pasos)
          → ROJO (CICLO_ROJO pasos)
          → VERDE ...

    Dos semáforos están coordinados: cuando uno está en VERDE, el otro está
    en ROJO (o AMARILLO transitando hacia ROJO). La coordinación la maneja
    el modelo, que crea ambos semáforos con fases opuestas.

    Atributos:
        eje (str)           : "HORIZONTAL" o "VERTICAL", solo informativo
        estado (str)        : estado actual (VERDE, AMARILLO o ROJO)
        _contador (int)     : pasos acumulados en el estado actual
        _duracion (int)     : pasos totales que debe durar el estado actual
    """

    def __init__(self, eje: str, estado_inicial: str):
        """
        Inicializa el semáforo.

        Parámetros:
            eje (str)            : "HORIZONTAL" o "VERTICAL"
            estado_inicial (str) : estado con el que arranca (VERDE o ROJO)
        """
        self.eje = eje                        # Eje que controla este semáforo
        self.estado = estado_inicial          # Estado inicial del semáforo
        self._contador = 0                    # Pasos transcurridos en el estado actual
        # Duración inicial según el estado de arranque
        self._duracion = self._duracion_de(estado_inicial)

    def _duracion_de(self, estado: str) -> int:
        """
        Retorna cuántos pasos dura el estado indicado.

        Parámetros:
            estado (str): VERDE, AMARILLO o ROJO

        Retorna:
            int: número de pasos que dura ese estado
        """
        if estado == VERDE:
            return CICLO_VERDE
        elif estado == AMARILLO:
            return CICLO_AMARILLO
        else:  # ROJO
            return CICLO_ROJO

    def _siguiente_estado(self) -> str:
        """
        Retorna el siguiente estado en el ciclo VERDE → AMARILLO → ROJO → VERDE.

        Retorna:
            str: nombre del siguiente estado
        """
        if self.estado == VERDE:
            return AMARILLO    # Después de verde siempre viene amarillo
        elif self.estado == AMARILLO:
            return ROJO        # Después de amarillo siempre viene rojo
        else:
            return VERDE       # Después de rojo vuelve a verde

    def step(self):
        """
        Avanza el semáforo un paso. Si se agota la duración del estado actual,
        transiciona al siguiente estado del ciclo.
        """
        self._contador += 1                          # Contamos un paso más

        if self._contador >= self._duracion:
            # Se acabó el tiempo del estado actual: transicionamos
            self.estado = self._siguiente_estado()   # Nuevo estado
            self._duracion = self._duracion_de(self.estado)  # Duración del nuevo estado
            self._contador = 0                       # Reiniciamos el contador

    def permite_avanzar(self) -> bool:
        """
        Indica si los carros controlados por este semáforo pueden cruzar.
        Solo el estado VERDE permite avanzar; AMARILLO y ROJO detienen el tráfico.

        Retorna:
            bool: True si el semáforo está en VERDE
        """
        return self.estado == VERDE

print("Clase Semaforo definida correctamente.")

## Celda 5 - Clase CarroAgent

In [ ]:
# =============================================================================
# CLASE: CarroAgent
# =============================================================================

class CarroAgent(Agent):
    """
    Representa un vehículo que circula en una de las 4 direcciones:
    NORTE, SUR, ESTE u OESTE.

    Cada carro tiene:
    - Una dirección fija (no cambia de carril)
    - Una posición normalizada (0.0 = origen, 1.0 = destino / salida)
    - Un estado: MOVIENDO o ESPERANDO

    En cada paso, el carro:
    1. Verifica si el semáforo de su eje lo detiene antes de la intersección
    2. Verifica si el carro de adelante está demasiado cerca (cola)
    3. Si puede avanzar, suma VELOCIDAD_CARRO a su posición
    4. Si supera 1.0 (salió de la pantalla), se marca para eliminación
    """

    def __init__(self, model, direccion: str):
        """
        Crea un carro en el extremo de entrada de su carril.

        Parámetros:
            model (TraficoModel) : modelo al que pertenece
            direccion (str)      : dirección de circulación
        """
        super().__init__(model)                    # Inicializa el agente Mesa

        self.direccion = direccion                 # Dirección del carro (fija)
        self.posicion = 0.0                        # Posición inicial: borde de entrada
        self.estado = "MOVIENDO"                   # Estado inicial: en movimiento
        self.activo = True                         # True mientras está en la pantalla

        # Estado del semáforo en el paso anterior. Se usa para detectar la
        # transición rojo/amarillo → verde y aplicar el arranque escalonado.
        self._semaforo_estado_anterior = None

        # Pasos que aún debe esperar antes de arrancar (arranque escalonado).
        # Cuando el semáforo cambia a verde, cada carro en la cola recibe un
        # retraso proporcional a su posición en la fila: el primero arranca
        # inmediatamente, el segundo espera 2 pasos, el tercero 4, etc.
        self._pasos_espera_arranque = 0

    def _obtener_semaforo(self) -> Semaforo:
        """
        Retorna el semáforo que controla el eje de este carro.
        - NORTE y SUR pertenecen al eje VERTICAL
        - ESTE y OESTE pertenecen al eje HORIZONTAL

        Retorna:
            Semaforo: objeto semáforo que controla a este carro
        """
        if self.direccion in ("NORTE", "SUR"):
            return self.model.semaforo_vertical     # Eje vertical
        else:
            return self.model.semaforo_horizontal   # Eje horizontal

    def _hay_carro_adelante(self) -> bool:
        """
        Comprueba si hay otro carro del mismo carril que esté demasiado cerca
        por delante. Esto implementa la cola: un carro no puede acercarse
        más de DISTANCIA_MIN al que tiene delante.

        Retorna:
            bool: True si debe detenerse por el carro de adelante
        """
        for otro in self.model.agents:
            # Solo comparamos contra carros activos del mismo carril
            if (
                not isinstance(otro, CarroAgent)    # Ignoramos no-carros
                or otro is self                     # Ignoramos a nosotros mismos
                or not otro.activo                  # Ignoramos carros inactivos
                or otro.direccion != self.direccion # Solo mismo carril
            ):
                continue

            # El carro "adelante" tiene mayor posición (más avanzado en el carril)
            diferencia = otro.posicion - self.posicion

            # Si está adelante y dentro de la distancia mínima → hay que detenerse
            if 0 < diferencia < DISTANCIA_MIN:
                return True

        return False  # No hay ningún carro bloqueando

    def step(self):
        """
        Comportamiento del carro en cada paso de simulación.

        Prioridades (en orden):
        1. Si ya salió de la pantalla (posicion > 1.0) → se desactiva
        2. Si el semáforo acaba de cambiar a verde → calcular retraso de arranque
           escalonado según posición en la cola
        3. Si aún está en espera de arranque escalonado → descontar un paso
        4. Si el semáforo está en rojo/amarillo Y el carro llegó a STOP_LINE → espera
        5. Si hay un carro adelante dentro de DISTANCIA_MIN → espera (cola)
        6. En cualquier otro caso → avanza VELOCIDAD_CARRO
        """

        # ── Caso 1: el carro ya salió de la pantalla ──────────────────────────
        if self.posicion > 1.0:
            self.activo = False    # Se marca inactivo para que el modelo lo elimine
            self.estado = "FUERA"
            return                 # No hay más acciones que tomar

        # ── Obtener estado actual del semáforo ────────────────────────────────
        semaforo = self._obtener_semaforo()
        estado_actual = semaforo.estado

        # ── Caso 2: el semáforo acaba de cambiar a VERDE ──────────────────────
        # Detectamos la transición comparando con el estado del paso anterior.
        # Solo aplicamos el retraso si el carro está en la zona de cola
        # (entre STOP_LINE e INTER_MIN), es decir, estaba detenido esperando.
        recien_cambio_a_verde = (
            estado_actual == VERDE
            and self._semaforo_estado_anterior != VERDE
            and self.posicion >= STOP_LINE
            and self.posicion <= INTER_MIN
        )

        if recien_cambio_a_verde:
            # Contamos cuántos carros del mismo carril están DELANTE de este
            # y también en la zona de cola (posicion entre STOP_LINE e INTER_MIN).
            carros_delante_en_cola = sum(
                1 for a in self.model.agents
                if (
                    isinstance(a, CarroAgent)
                    and a is not self
                    and a.activo
                    and a.direccion == self.direccion
                    and a.posicion > self.posicion      # Está delante
                    and a.posicion <= INTER_MIN         # También en la cola
                )
            )
            # Retraso: 2 pasos por cada carro que tiene delante en la cola.
            self._pasos_espera_arranque = carros_delante_en_cola * 2

        # Guardamos el estado actual para detectar la transición en el siguiente paso
        self._semaforo_estado_anterior = estado_actual

        # ── Caso 3: aún en espera de arranque escalonado ──────────────────────
        if self._pasos_espera_arranque > 0:
            self._pasos_espera_arranque -= 1   # Descontamos un paso de espera
            self.estado = "ESPERANDO"
            return                             # No avanza todavía

        # ── Caso 4: semáforo en rojo/amarillo y carro en la línea de stop ─────
        semaforo_bloqueante = (
            not semaforo.permite_avanzar()           # El semáforo NO está en verde
            and self.posicion >= STOP_LINE           # El carro llegó a la línea de stop
            and self.posicion <= INTER_MIN           # El carro AÚN NO entró a la intersección
        )

        if semaforo_bloqueante:
            self.estado = "ESPERANDO"   # El carro espera en la línea de stop
            return                      # No avanza en este paso

        # ── Caso 5: hay un carro adelante muy cerca (respeto de cola) ─────────
        if self._hay_carro_adelante():
            self.estado = "ESPERANDO"   # El carro espera detrás del de adelante
            return                      # No avanza en este paso

        # ── Caso 6: puede avanzar libremente ──────────────────────────────────
        self.posicion += VELOCIDAD_CARRO    # Mueve el carro hacia adelante
        self.estado = "MOVIENDO"            # El carro está en movimiento

print("Clase CarroAgent definida correctamente.")

## Celda 6 - Clase TraficoModel

In [ ]:
# =============================================================================
# CLASE: TraficoModel
# =============================================================================

class TraficoModel(Model):
    """
    Modelo de tráfico. Gestiona los semáforos, los carros y las estadísticas.

    Responsabilidades:
    - Crear y actualizar los dos semáforos coordinados
    - Generar nuevos carros periódicamente en los bordes de cada carril
    - Eliminar carros que han salido de la pantalla
    - Recolectar datos de conteo de carros activos por dirección
    """

    def __init__(self, seed: int = 42):
        """
        Inicializa el modelo con semáforos y sin carros (se generan en step).

        Parámetros:
            seed (int): semilla para reproducibilidad de la simulación
        """
        super().__init__(rng=seed)   # Constructor Mesa con semilla

        # Paso actual de la simulación (se incrementa en cada step)
        self.paso_actual = 0

        # ── Semáforos coordinados ────────────────────────────────────────────
        # El eje horizontal arranca en VERDE → los carros E-O pueden pasar
        self.semaforo_horizontal = Semaforo("HORIZONTAL", VERDE)

        # El eje vertical arranca en ROJO → los carros N-S deben esperar
        self.semaforo_vertical = Semaforo("VERTICAL", ROJO)

        # ── Offsets de spawn por dirección ────────────────────────────────────
        # Cada dirección recibe un desplazamiento aleatorio dentro del intervalo
        # de spawn. Así los carros de cada carril aparecen en pasos distintos
        # en lugar de todos al mismo tiempo, evitando colisiones iniciales.
        self._spawn_offsets = {
            d: random.randint(0, SPAWN_INTERVALO - 1)
            for d in ("NORTE", "SUR", "ESTE", "OESTE")
        }

        # ── DataCollector ─────────────────────────────────────────────────────
        # Registra en cada paso cuántos carros activos hay por dirección
        self.datacollector = DataCollector(
            model_reporters={
                # Carros activos que van hacia el NORTE
                "Norte": lambda m: sum(
                    1 for a in m.agents
                    if isinstance(a, CarroAgent) and a.activo and a.direccion == "NORTE"
                ),
                # Carros activos que van hacia el SUR
                "Sur": lambda m: sum(
                    1 for a in m.agents
                    if isinstance(a, CarroAgent) and a.activo and a.direccion == "SUR"
                ),
                # Carros activos que van hacia el ESTE
                "Este": lambda m: sum(
                    1 for a in m.agents
                    if isinstance(a, CarroAgent) and a.activo and a.direccion == "ESTE"
                ),
                # Carros activos que van hacia el OESTE
                "Oeste": lambda m: sum(
                    1 for a in m.agents
                    if isinstance(a, CarroAgent) and a.activo and a.direccion == "OESTE"
                ),
                # Estado de los semáforos (para estadísticas)
                "Semaforo_H": lambda m: m.semaforo_horizontal.estado,
                "Semaforo_V": lambda m: m.semaforo_vertical.estado,
            }
        )

        # Recolectamos datos del estado inicial (sin carros)
        self.datacollector.collect(self)

    def _contar_carros_activos(self) -> int:
        """
        Cuenta cuántos carros activos hay en este momento.

        Retorna:
            int: número de CarroAgent con activo=True
        """
        return sum(
            1 for a in self.agents
            if isinstance(a, CarroAgent) and a.activo
        )

    def _generar_carro(self, direccion: str):
        """
        Intenta crear un nuevo carro en el extremo de entrada del carril indicado.
        No lo crea si:
          - Ya hay un carro cerca del punto de entrada (evita solapamiento al nacer), o
          - El carril ya tiene MAX_CARROS_POR_CARRIL carros activos.

        Parámetros:
            direccion (str): dirección del nuevo carro
        """
        # Contamos cuántos carros activos hay ya en este carril
        carros_en_carril = sum(
            1 for a in self.agents
            if isinstance(a, CarroAgent) and a.activo and a.direccion == direccion
        )

        # Si el carril ya alcanzó el límite por dirección, no generamos más
        if carros_en_carril >= MAX_CARROS_POR_CARRIL:
            return

        # Verificamos que no haya un carro muy cerca del punto de entrada
        hay_carro_en_entrada = any(
            a for a in self.agents
            if (
                isinstance(a, CarroAgent)
                and a.activo
                and a.direccion == direccion
                and a.posicion < DISTANCIA_MIN
            )
        )

        if not hay_carro_en_entrada:
            # Creamos el nuevo carro
            CarroAgent(self, direccion)

    def step(self):
        """
        Ejecuta un paso completo de la simulación:

        1. Avanzamos el contador de pasos
        2. Actualizamos los semáforos (pueden cambiar de estado)
        3. Generamos nuevos carros en los bordes si toca según el intervalo
        4. Ejecutamos el step() de cada carro activo (en orden de posición,
           del más adelantado al más atrasado, para respetar la cola)
        5. Eliminamos los carros que ya salieron de la pantalla
        6. Recolectamos estadísticas
        """

        # ── 1. Avanzamos el contador global ───────────────────────────────────
        self.paso_actual += 1

        # ── 2. Actualizamos los semáforos ─────────────────────────────────────
        self.semaforo_horizontal.step()
        self.semaforo_vertical.step()

        # ── 3. Generamos nuevos carros si corresponde ─────────────────────────
        for direccion in ("NORTE", "SUR", "ESTE", "OESTE"):
            offset = self._spawn_offsets[direccion]
            if (self.paso_actual + offset) % SPAWN_INTERVALO == 0:
                self._generar_carro(direccion)

        # ── 4. Ejecutamos cada carro en orden (adelante → atrás) ──────────────
        carros_ordenados = sorted(
            [a for a in self.agents if isinstance(a, CarroAgent) and a.activo],
            key=lambda c: c.posicion,
            reverse=True   # Mayor posición primero (el más avanzado)
        )

        for carro in carros_ordenados:
            carro.step()

        # ── 5. Eliminamos carros inactivos ────────────────────────────────────
        carros_inactivos = [
            a for a in self.agents
            if isinstance(a, CarroAgent) and not a.activo
        ]
        for carro in carros_inactivos:
            carro.remove()

        # ── 6. Recolectamos estadísticas del paso ──────────────────────────────
        self.datacollector.collect(self)

print("Clase TraficoModel definida correctamente.")

## Celda 7 - Función de animación y resumen estadístico

In [ ]:
# =============================================================================
# FUNCIÓN DE ANIMACIÓN
# =============================================================================

def animar_simulacion(num_pasos: int = 120, intervalo_ms: int = 120):
    """
    Construye la animación de la simulación de tráfico.

    La figura tiene dos paneles:
    - Izquierdo (intersección): vista aérea con carros moviéndose en sus
      carriles y semáforos cambiando de color.
    - Derecho (barras): conteo de carros activos por dirección.

    En Google Colab la animación se renderiza como HTML embebido con
    controles de reproducción interactivos (play/pause/loop).

    Parámetros:
        num_pasos (int)    : número de pasos/frames de la simulación
        intervalo_ms (int) : milisegundos entre frames

    Retorna:
        tuple (FuncAnimation, TraficoModel): animación y modelo con los datos
    """

    # Creamos el modelo (sin carros al inicio; se generan en step)
    modelo = TraficoModel(seed=42)

    # ── Configuración de la figura ────────────────────────────────────────────

    # Figura con dos subplots: intersección (ancho 2) y barras (ancho 1)
    fig, (ax_inter, ax_barras) = plt.subplots(
        1, 2,
        figsize=(16, 8),
        gridspec_kw={"width_ratios": [2, 1]}
    )

    # Título general de la figura
    fig.suptitle(
        "Simulación de Tráfico en una Intersección con Semáforos",
        fontsize=15, fontweight="bold", y=0.98
    )

    # ─────────────────────────────────────────────────────────────────────────
    # PANEL IZQUIERDO: Vista aérea de la intersección
    # ─────────────────────────────────────────────────────────────────────────

    ax_inter.set_xlim(0, 1)
    ax_inter.set_ylim(0, 1)
    ax_inter.set_aspect("equal")
    ax_inter.axis("off")

    # ── Fondo verde (hierba) en las 4 esquinas ───────────────────────────────
    for (x0, y0, ancho, alto) in [
        (0.0,       0.0,       INTER_MIN,     INTER_MIN),
        (INTER_MAX, 0.0,       1-INTER_MAX,   INTER_MIN),
        (0.0,       INTER_MAX, INTER_MIN,     1-INTER_MAX),
        (INTER_MAX, INTER_MAX, 1-INTER_MAX,   1-INTER_MAX),
    ]:
        rect_verde = mpatches.Rectangle(
            (x0, y0), ancho, alto,
            facecolor="#4CAF50",
            edgecolor="none",
            zorder=1
        )
        ax_inter.add_patch(rect_verde)

    # ── Calles (franjas grises) ───────────────────────────────────────────────
    calle_h = mpatches.Rectangle(
        (0.0, INTER_MIN),
        1.0, INTER_MAX - INTER_MIN,
        facecolor="#555555",
        edgecolor="none",
        zorder=2
    )
    ax_inter.add_patch(calle_h)

    calle_v = mpatches.Rectangle(
        (INTER_MIN, 0.0),
        INTER_MAX - INTER_MIN, 1.0,
        facecolor="#555555",
        edgecolor="none",
        zorder=2
    )
    ax_inter.add_patch(calle_v)

    # ── Líneas de carril (divisorias amarillas punteadas) ────────────────────
    ax_inter.plot(
        [0.0, INTER_MIN], [0.5, 0.5],
        color="#FFD700", linewidth=1.5, linestyle="--", zorder=3
    )
    ax_inter.plot(
        [INTER_MAX, 1.0], [0.5, 0.5],
        color="#FFD700", linewidth=1.5, linestyle="--", zorder=3
    )
    ax_inter.plot(
        [0.5, 0.5], [0.0, INTER_MIN],
        color="#FFD700", linewidth=1.5, linestyle="--", zorder=3
    )
    ax_inter.plot(
        [0.5, 0.5], [INTER_MAX, 1.0],
        color="#FFD700", linewidth=1.5, linestyle="--", zorder=3
    )

    # ── Zona de intersección (cuadrado central) ───────────────────────────────
    inter_rect = mpatches.Rectangle(
        (INTER_MIN, INTER_MIN),
        INTER_MAX - INTER_MIN, INTER_MAX - INTER_MIN,
        facecolor="#666666",
        edgecolor="none",
        zorder=3
    )
    ax_inter.add_patch(inter_rect)

    # Paso de cebra (líneas blancas semitransparentes)
    num_rayas = 5
    ancho_raya = (INTER_MAX - INTER_MIN) / (num_rayas * 2 - 1)
    for i in range(num_rayas):
        x_raya = INTER_MIN + i * ancho_raya * 2
        cebra = mpatches.Rectangle(
            (x_raya, INTER_MIN),
            ancho_raya, INTER_MAX - INTER_MIN,
            facecolor="white", alpha=0.25, edgecolor="none", zorder=4
        )
        ax_inter.add_patch(cebra)

    # ── Etiquetas de dirección en los extremos de cada carril ────────────────
    etiquetas_dir = [
        (0.5,  0.97, "↓ SUR",   COLOR_CARRO["SUR"]),
        (0.5,  0.03, "↑ NORTE", COLOR_CARRO["NORTE"]),
        (0.03, 0.5,  "→ ESTE",  COLOR_CARRO["ESTE"]),
        (0.97, 0.5,  "← OESTE", COLOR_CARRO["OESTE"]),
    ]
    for (x, y, texto, color) in etiquetas_dir:
        ax_inter.text(
            x, y, texto,
            ha="center", va="center",
            fontsize=9, fontweight="bold", color=color,
            zorder=6
        )

    # ── Semáforos visuales ────────────────────────────────────────────────────

    # Semáforo horizontal
    caja_semaforo_h = mpatches.FancyBboxPatch(
        (INTER_MIN - 0.065, 0.5 - 0.025), 0.055, 0.05,
        boxstyle="round,pad=0.005",
        facecolor="#1a1a1a", edgecolor="#333333", linewidth=1,
        zorder=7
    )
    ax_inter.add_patch(caja_semaforo_h)

    circulo_semaforo_h = mpatches.Circle(
        (INTER_MIN - 0.037, 0.5), 0.016,
        facecolor="green",
        edgecolor="none",
        zorder=8
    )
    ax_inter.add_patch(circulo_semaforo_h)

    ax_inter.text(
        INTER_MIN - 0.037, 0.5 + 0.038,
        "E-O", ha="center", va="bottom",
        fontsize=7, color="white", zorder=8
    )

    # Semáforo vertical
    caja_semaforo_v = mpatches.FancyBboxPatch(
        (0.5 - 0.025, INTER_MIN - 0.065), 0.05, 0.055,
        boxstyle="round,pad=0.005",
        facecolor="#1a1a1a", edgecolor="#333333", linewidth=1,
        zorder=7
    )
    ax_inter.add_patch(caja_semaforo_v)

    circulo_semaforo_v = mpatches.Circle(
        (0.5, INTER_MIN - 0.037), 0.016,
        facecolor="red",
        edgecolor="none",
        zorder=8
    )
    ax_inter.add_patch(circulo_semaforo_v)

    ax_inter.text(
        0.5, INTER_MIN - 0.072,
        "N-S", ha="center", va="top",
        fontsize=7, color="white", zorder=8
    )

    # ── Scatter de carros ────────────────────────────────────────────────────
    # Se inicializa con np.empty((0, 2)) para garantizar la forma 2D correcta
    # desde el primer frame y evitar el IndexError con arrays vacíos.
    scatter_carros = ax_inter.scatter(
        np.empty((0, 2))[:, 0], np.empty((0, 2))[:, 1],
        s=120,
        marker="s",     # Cuadrado para simular un carro desde arriba
        zorder=9,
        edgecolors="#1a1a1a",
        linewidths=0.8
    )

    # ── Textos del panel de intersección ────────────────────────────────────
    texto_paso = ax_inter.text(
        0.02, 0.02,
        "Paso: 0",
        transform=ax_inter.transAxes,
        fontsize=11, color="white", fontweight="bold", zorder=10
    )

    texto_semaforos = ax_inter.text(
        0.5, 0.02, "",
        transform=ax_inter.transAxes,
        fontsize=9, color="white",
        ha="center", va="bottom", zorder=10
    )

    texto_conteo = ax_inter.text(
        0.5, 0.97, "",
        transform=ax_inter.transAxes,
        fontsize=9, color="white",
        ha="center", va="top", zorder=10
    )

    # ─────────────────────────────────────────────────────────────────────────
    # PANEL DERECHO: Barras de conteo por dirección
    # ─────────────────────────────────────────────────────────────────────────

    nombres_dir = ["Norte", "Sur", "Este", "Oeste"]
    x_pos = list(range(4))
    colores_barras = [
        COLOR_CARRO["NORTE"],
        COLOR_CARRO["SUR"],
        COLOR_CARRO["ESTE"],
        COLOR_CARRO["OESTE"],
    ]

    barras = ax_barras.bar(
        x_pos, [0, 0, 0, 0],
        color=colores_barras,
        alpha=0.85, zorder=2
    )

    ax_barras.set_xticks(x_pos)
    ax_barras.set_xticklabels(nombres_dir, fontsize=11)
    ax_barras.set_ylim(0, MAX_CARROS_POR_CARRIL + 2)
    ax_barras.set_ylabel("Carros activos", fontsize=11)
    ax_barras.set_title("Carros activos por dirección", fontsize=11, fontweight="bold")
    ax_barras.grid(True, axis="y", alpha=0.3, zorder=0)
    ax_barras.set_facecolor("#FAFAFA")

    etiquetas_barras = []
    for i in range(4):
        etq = ax_barras.text(
            x_pos[i], 0, "",
            ha="center", va="bottom",
            fontsize=13, fontweight="bold", color="#2C3E50"
        )
        etiquetas_barras.append(etq)

    texto_semaforo_barras = ax_barras.text(
        0.5, 0.97, "",
        transform=ax_barras.transAxes,
        fontsize=9, ha="center", va="top",
        color="#333333"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # ─────────────────────────────────────────────────────────────────────────
    # Mapa de colores de semáforo para los círculos visuales
    # ─────────────────────────────────────────────────────────────────────────
    COLOR_SEMAFORO = {
        VERDE:    "#00CC44",
        AMARILLO: "#FFD700",
        ROJO:     "#FF3333",
    }

    # ─────────────────────────────────────────────────────────────────────────
    # Función de actualización por frame
    # ─────────────────────────────────────────────────────────────────────────

    def actualizar(frame):
        """
        Se ejecuta una vez por frame. Avanza el modelo un paso y actualiza
        todos los elementos visuales: scatter de carros, semáforos y barras.
        """

        # El frame 0 muestra el estado inicial sin ejecutar el modelo
        if frame > 0:
            modelo.step()

        # ── Posiciones y colores de los carros activos ────────────────────────
        xs = []
        ys = []
        cols = []

        for carro in modelo.agents:
            if not isinstance(carro, CarroAgent) or not carro.activo:
                continue

            # Convertimos la posición lógica [0,1] a coordenadas visuales (x, y)
            # según la dirección de circulación.
            #
            # Eje VERTICAL (x = 0.50):
            #   Carril SUR   → x = 0.59  (derecha del centro, va hacia abajo)
            #   Carril NORTE → x = 0.41  (izquierda del centro, va hacia arriba)
            #
            # Eje HORIZONTAL (y = 0.50):
            #   Carril ESTE  → y = 0.59  (arriba del centro, va hacia la derecha)
            #   Carril OESTE → y = 0.41  (abajo del centro, va hacia la izquierda)

            if carro.direccion == "SUR":
                x = 0.59
                y = 1.0 - carro.posicion
            elif carro.direccion == "NORTE":
                x = 0.41
                y = carro.posicion
            elif carro.direccion == "ESTE":
                x = carro.posicion
                y = 0.59
            else:  # "OESTE"
                x = 1.0 - carro.posicion
                y = 0.41

            xs.append(x)
            ys.append(y)
            cols.append(COLOR_CARRO[carro.direccion])

        # Actualizamos el scatter con las posiciones y colores nuevos.
        # Con carros presentes usamos np.column_stack para obtener un array (N, 2).
        # Sin carros usamos np.empty((0, 2)) para mantener la forma 2D y evitar
        # el IndexError que ocurre si se pasa una lista vacía [] (array 1D de shape (0,)).
        if xs:
            offsets = np.column_stack([xs, ys])
        else:
            offsets = np.empty((0, 2))

        scatter_carros.set_offsets(offsets)
        scatter_carros.set_color(cols)

        # ── Actualizamos el color de los círculos de semáforo ─────────────────
        estado_h = modelo.semaforo_horizontal.estado
        estado_v = modelo.semaforo_vertical.estado

        circulo_semaforo_h.set_facecolor(COLOR_SEMAFORO[estado_h])
        circulo_semaforo_v.set_facecolor(COLOR_SEMAFORO[estado_v])

        # ── Textos informativos ───────────────────────────────────────────────
        texto_paso.set_text(f"Paso: {frame}  /  {num_pasos}")

        texto_semaforos.set_text(
            f"E-O: {estado_h}   |   N-S: {estado_v}"
        )

        n_norte = sum(
            1 for a in modelo.agents
            if isinstance(a, CarroAgent) and a.activo and a.direccion == "NORTE"
        )
        n_sur = sum(
            1 for a in modelo.agents
            if isinstance(a, CarroAgent) and a.activo and a.direccion == "SUR"
        )
        n_este = sum(
            1 for a in modelo.agents
            if isinstance(a, CarroAgent) and a.activo and a.direccion == "ESTE"
        )
        n_oeste = sum(
            1 for a in modelo.agents
            if isinstance(a, CarroAgent) and a.activo and a.direccion == "OESTE"
        )

        texto_conteo.set_text(
            f"↑N:{n_norte}  ↓S:{n_sur}  →E:{n_este}  ←O:{n_oeste}"
            f"  (Total: {n_norte+n_sur+n_este+n_oeste})"
        )

        # ── Actualizamos las barras del panel derecho ─────────────────────────
        valores = [n_norte, n_sur, n_este, n_oeste]
        for i, (barra, valor) in enumerate(zip(barras, valores)):
            barra.set_height(valor)
            etiquetas_barras[i].set_text(str(valor))
            etiquetas_barras[i].set_y(valor + 0.1)

        texto_semaforo_barras.set_text(
            f"Semáforo E-O: {estado_h}   |   Semáforo N-S: {estado_v}"
        )

        # Retornamos los artistas modificados
        return (
            [scatter_carros, texto_paso, texto_semaforos, texto_conteo,
             circulo_semaforo_h, circulo_semaforo_v, texto_semaforo_barras]
            + list(barras)
            + etiquetas_barras
        )

    # ── Creamos la animación ───────────────────────────────────────────────────
    anim = FuncAnimation(
        fig,
        actualizar,
        frames=num_pasos + 1,
        interval=intervalo_ms,
        repeat=True,
        blit=False          # blit=False para compatibilidad con Colab
    )

    # Cerramos la figura estática para que no aparezca duplicada en el notebook
    plt.close(fig)

    # Retornamos la animación Y el modelo (necesario para imprimir_resumen)
    return anim, modelo


# =============================================================================
# FUNCIÓN DE RESUMEN
# =============================================================================

def imprimir_resumen(modelo: TraficoModel, num_pasos: int):
    """
    Imprime estadísticas finales de la simulación en consola.

    Parámetros:
        modelo (TraficoModel) : el modelo ya ejecutado
        num_pasos (int)       : número de pasos simulados
    """
    datos = modelo.datacollector.get_model_vars_dataframe()

    print("\n" + "=" * 58)
    print("   RESUMEN DE LA SIMULACIÓN - TRÁFICO EN INTERSECCIÓN")
    print("=" * 58)
    print(f"  Pasos simulados       : {num_pasos}")
    print(f"  Velocidad de carros   : {VELOCIDAD_CARRO} unidades/paso")
    print(f"  Ciclo verde           : {CICLO_VERDE} pasos")
    print(f"  Ciclo amarillo        : {CICLO_AMARILLO} pasos")
    print(f"  Intervalo de spawn    : cada {SPAWN_INTERVALO} pasos")
    print(f"  Máx. carros por carril: {MAX_CARROS_POR_CARRIL}")
    print()
    print("  Estadísticas de carros activos por dirección:")
    print("  " + "-" * 52)
    print(f"  {'Dirección':<12} {'Promedio':>10} {'Máximo':>10} {'Final':>10}")
    print("  " + "-" * 52)

    for col in ["Norte", "Sur", "Este", "Oeste"]:
        promedio = datos[col].mean()
        maximo   = datos[col].max()
        final    = datos[col].iloc[-1]
        print(f"  {col:<12} {promedio:>10.1f} {maximo:>10} {final:>10}")

    print("=" * 58 + "\n")

print("Funciones animar_simulacion e imprimir_resumen definidas correctamente.")

## Celda 8 - Ejecución: animación y resumen

Esta celda lanza la simulación completa y muestra la animación como un reproductor HTML interactivo con controles play/pause/loop.

> **Nota:** la generación del HTML puede tardar 10-20 segundos ya que pre-renderiza los 120 frames antes de mostrarse.

In [ ]:
# =============================================================================
# EJECUCIÓN PRINCIPAL
# =============================================================================

# Número de pasos/frames de la animación
NUM_PASOS    = 120

# Milisegundos entre frames (120ms ≈ 8fps, ritmo fluido y legible)
INTERVALO_MS = 120

print("\n" + "=" * 58)
print("  SIMULACIÓN DE TRÁFICO EN UNA INTERSECCIÓN")
print("=" * 58)
print(f"  Pasos           : {NUM_PASOS}")
print(f"  Intervalo       : {INTERVALO_MS} ms por frame")
print(f"  Ciclo semáforo  : {CICLO_VERDE}v / {CICLO_AMARILLO}a / {CICLO_ROJO}r pasos")
print(f"  Spawn cada      : {SPAWN_INTERVALO} pasos por dirección")
print(f"  Máx. carros por carril: {MAX_CARROS_POR_CARRIL}")
print("=" * 58)
print("Generando animación... (puede tardar 10-20 segundos)")

# Generamos la animación y obtenemos el modelo con los datos recolectados
anim, modelo_final = animar_simulacion(
    num_pasos=NUM_PASOS,
    intervalo_ms=INTERVALO_MS
)

# Mostramos la animación como HTML embebido en el notebook.
# to_jshtml() genera un reproductor interactivo con controles play/pause/loop.
display(HTML(anim.to_jshtml()))

# Resumen estadístico de la simulación
imprimir_resumen(modelo_final, NUM_PASOS)